In [6]:
# scripts/build_instance_prior.py
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from scipy.signal import butter, filtfilt, resample_poly

# ----------------------- 配置 -----------------------
ROOT = Path(r"D:\Project_Github\audio_click_mil")

AUDIO_DIR = ROOT / "data" / "original_audio"
BAG_CSV = ROOT / "processed_data" / "all_bags.csv"

OUTPUT_PATH = ROOT / "processed_data" / "instance_prior.csv"

EXCLUDE_FILES = {12}
TARGET_SR = 48000

# ----------------------- 工具函数 -----------------------
def parse_file_num(filename: str) -> int:
    return int(Path(filename).stem[-2:])


def highpass_filter(signal, sr, cutoff=5000, order=6):
    """
    6阶巴特沃斯高通滤波器
    """
    nyquist = 0.5 * sr
    norm_cutoff = cutoff / nyquist
    b, a = butter(order, norm_cutoff, btype='high', analog=False)
    return filtfilt(b, a, signal)


def resample_audio(signal, orig_sr, target_sr):
    """
    使用 polyphase 方法降采样（抗混叠）
    """
    if orig_sr == target_sr:
        return signal
    return resample_poly(signal, target_sr, orig_sr)


def compute_tkeo(signal: np.ndarray):
    if len(signal) < 3:
        return np.zeros_like(signal)
    x = signal
    return x[1:-1]**2 - x[:-2] * x[2:]


def tkeo_max_1s(signal: np.ndarray):
    tkeo = compute_tkeo(signal)
    if len(tkeo) == 0:
        return 0.0
    return float(np.max(tkeo)) #负值由（数值/滤波/相位导致），只取正能量，更符合物理意义


# ----------------------- 主流程 -----------------------
def build_instance_prior():
    print("读取 bag 信息...")
    bag_df = pd.read_csv(BAG_CSV)

    all_results = []

    # 按文件处理（用于 per-file 归一化）
    for file_num, file_df in tqdm(bag_df.groupby("file_num")):

        if file_num in EXCLUDE_FILES:
            continue

        audio_filename = file_df["audio_file"].iloc[0]
        audio_path = AUDIO_DIR / audio_filename

        # -----------------------
        # Step 0: 读取音频
        # -----------------------
        try:
            audio, sr = sf.read(str(audio_path))
        except Exception as e:
            print(f"读取失败 {audio_filename}: {e}")
            continue

        # 转单通道
        if audio.ndim > 1:
            audio = np.mean(audio, axis=1)

        # -----------------------
        # Step 1: 降采样到48kHz
        # -----------------------
        audio = resample_audio(audio, sr, TARGET_SR)
        sr = TARGET_SR

        # -----------------------
        # Step 2: 高通滤波（5kHz）
        # -----------------------
        try:
            audio = highpass_filter(audio, sr)
        except Exception as e:
            print(f"滤波失败 {audio_filename}: {e}")
            continue

        # -----------------------
        # Step 3: 逐秒计算TKEO
        # -----------------------
        duration = len(audio) / sr
        total_seconds = int(duration)

        instance_scores = []

        for sec in range(total_seconds):
            start = int(sec * sr)
            end = int((sec + 1) * sr)

            segment = audio[start:end]
            score = tkeo_max_1s(segment)
            instance_scores.append(score)

        instance_scores = np.array(instance_scores)

        # -----------------------
        # Step 4: per-file 归一化
        # -----------------------
        min_v = instance_scores.min()
        max_v = instance_scores.max()

        if max_v - min_v < 1e-8:
            norm_scores = np.zeros_like(instance_scores)
        else:
            norm_scores = (instance_scores - min_v) / (max_v - min_v)

        # -----------------------
        # Step 5: 对齐 bag
        # -----------------------
        for _, row in file_df.iterrows():
            bag_idx = int(row["bag_idx"])
            bag_start = bag_idx * 60

            for i in range(60):
                instance_idx = i
                global_sec = bag_start + i

                if global_sec >= len(norm_scores):
                    continue

                all_results.append({
                    "file_num": file_num,
                    "bag_idx": bag_idx,
                    "instance_idx": instance_idx,
                    "prior_score": float(norm_scores[global_sec])
                })

    # -----------------------
    # 保存
    # -----------------------
    df_out = pd.DataFrame(all_results)
    df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

    print(f"\n完成！共生成 {len(df_out)} 条 instance 记录")
    print(f"保存路径: {OUTPUT_PATH}")


if __name__ == "__main__":
    build_instance_prior()

读取 bag 信息...


100%|██████████| 34/34 [07:38<00:00, 13.48s/it]



完成！共生成 57420 条 instance 记录
保存路径: D:\Project_Github\audio_click_mil\processed_data\instance_prior.csv
